In [2]:

# MLOps Exercise 01: Reproducible Decision Tree Experiments

import pandas as pd
import json
from pathlib import Path

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)
from joblib import dump

# 1. Experiment configuration
RANDOM_STATE = 42
TEST_SIZE = 0.20
DEPTHS = [1, 2, 3, 5, None]

# 2. Load the dataset
iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

print("Dataset shape:", X.shape)
print("Features:", list(X.columns))
print("Classes:", list(iris.target_names))

# 3. Create a reproducible train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

# 4. Train and evaluate multiple experiments
results = []
best_model = None
best_f1 = -1
best_depth = None

for depth in DEPTHS:
    model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=RANDOM_STATE
    )
    model.fit(X_train, y_train)

    train_pred = model.predict(X_train)
    test_pred = model.predict(X_test)

    test_f1 = f1_score(
        y_test, test_pred, average="macro"
    )

    results.append({
        "experiment_id": f"tree_depth_{depth}",
        "algorithm": "DecisionTree",
        "max_depth": str(depth),
        "random_state": RANDOM_STATE,
        "test_size": TEST_SIZE,
        "train_accuracy": accuracy_score(
            y_train, train_pred
        ),
        "test_accuracy": accuracy_score(
            y_test, test_pred
        ),
        "precision_macro": precision_score(
            y_test, test_pred, average="macro",
            zero_division=0
        ),
        "recall_macro": recall_score(
            y_test, test_pred, average="macro",
            zero_division=0
        ),
        "f1_macro": test_f1
    })

    if test_f1 > best_f1:
        best_f1 = test_f1
        best_model = model
        best_depth = depth

# 5. Save experiment results
results_df = pd.DataFrame(results)
results_df.to_csv("experiment_results.csv", index=False)

print("\nExperiment comparison:")
display(results_df.round(4))

# 6. Evaluate the selected model
best_predictions = best_model.predict(X_test)

print("Selected depth:", best_depth)
print("Confusion matrix:")
print(confusion_matrix(y_test, best_predictions))

# 7. Save model and experiment configuration
dump(best_model, "best_model.joblib")

config = {
    "dataset": "Iris",
    "algorithm": "DecisionTreeClassifier",
    "random_state": RANDOM_STATE,
    "test_size": TEST_SIZE,
    "tested_depths": [str(d) for d in DEPTHS],
    "selection_metric": "macro_f1",
    "selected_depth": str(best_depth)
}

Path("experiment_config.json").write_text(
    json.dumps(config, indent=2)
)

print("\nFiles saved:")
print("- experiment_results.csv")
print("- best_model.joblib")
print("- experiment_config.json")



Dataset shape: (150, 4)
Features: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
Classes: [np.str_('setosa'), np.str_('versicolor'), np.str_('virginica')]

Experiment comparison:


,experiment_id,algorithm,max_depth,random_state,test_size,train_accuracy,test_accuracy,precision_macro,recall_macro,f1_macro
0,tree_depth_1,DecisionTree,1,42,0.2,0.6667,0.6667,0.5000,0.6667,0.5556
1,tree_depth_2,DecisionTree,2,42,0.2,0.9667,0.9333,0.9333,0.9333,0.9333
2,tree_depth_3,DecisionTree,3,42,0.2,0.9833,0.9667,0.9697,0.9667,0.9666
3,tree_depth_5,DecisionTree,5,42,0.2,1.0000,0.9333,0.9333,0.9333,0.9333
4,tree_depth_None,DecisionTree,None,42,0.2,1.0000,0.9333,0.9333,0.9333,0.9333


Selected depth: 3
Confusion matrix:
[[10  0  0]
 [ 0  9  1]
 [ 0  0 10]]

Files saved:
- experiment_results.csv
- best_model.joblib
- experiment_config.json
